# PRJEB30386 — эталон после аннотации → PCR1 → фрагментация → PCR2 → NovaSeq PE150

Биологический эталон строится только из `post_annotation_filtered/airr_pass`. Симуляция сохраняет наблюдаемую нуклеотидную последовательность V…J, включая реальные SHM, использует численность UMI как число исходных копий, добавляет общую модель ошибок InSilicoSeq NovaSeq.

**Фрагментация:** `uniform single random-cut`.


## Выходные данные

Все создаваемые артефакты сохраняются в ветви:

`results/PRJEB30386/simulated/insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut/`

Каталог `qc/` предназначен для контроля качества этой симуляции.


## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))), None)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

# Для ветви NovaSeq требуется только InSilicoSeq; отсутствие сторонних
# инструментов выравнивания не является ошибкой.
for tool in ("iss",):
    p = shutil.which(tool)
    if not p:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {p}")
print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut"
SOURCE_STAGE = "post_annotation_filtered"

VOLUME = Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin"))
if not (VOLUME / "results/PRJEB30386").exists():
    found = None
    for start in (Path.cwd().resolve(), Path("/Users/epishkin/workspace/bcr-assembler")):
        for candidate in (start, *start.parents):
            if (candidate / "results/PRJEB30386").exists() and (candidate / "scripts").is_dir():
                found = candidate
                break
        if found:
            break
    if not found:
        raise FileNotFoundError("Cannot locate bcr-assembler root; set BCR_VOLUME")
    VOLUME = found

DATASET = "PRJEB30386"
SOURCE_RUNS = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"]
RUN_LOCUS = {
    "ERR3004229": "IGH",
    "ERR3004230": "IGH",
    "ERR3004231": "IGK",
    "ERR3004232": "IGL",
}
RUN_LIBRARY = {
    "ERR3004229": "IgM",
    "ERR3004230": "IgG",
    "ERR3004231": "IgK",
    "ERR3004232": "IgL",
}
MIXED_SAMPLE = "PRJEB30386_all_chains"
SAMPLES = [MIXED_SAMPLE]

DATASET_DIR = VOLUME / "results" / DATASET
POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / DATASET
PR_FASTQ_DIR = DATASET_DIR / "pr_trimmed" / "fastq"
ORIGINAL_AIRR_DIR = DATASET_DIR / "annotation" / "igblast"

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run filter_human_post_annotation.ipynb first."
    )
FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS
MIN_TEMPLATE_LENGTH = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
TEMPLATES_DIR = TRUTH_DIR
COUNTS_DIR = ALLOCATION_DIR

for d in (
    TRUTH_DIR, PCR1_DIR, FRAGMENTATION_DIR, PCR2_DIR, ALLOCATION_DIR,
    FASTQ_NATIVE_DIR, FASTQ_DIR, MODEL_DIR, LOGS_DIR, QC_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

NPROC = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False
COMPRESS = True

# Дополнительные требования симуляции к эталонным данным после
# основной фильтрации аннотаций.
DROP_SEQUENCES_WITH_N = True
REQUIRE_VALID_20NT_UMI = True
STARTING_COPIES_MODE = "unique_umis"

# PCR1: стохастическое ветвление на каждом цикле.
PCR_CYCLES = 25
PCR_EFFICIENCY_MEAN = 0.85
PCR_EFFICIENCY_CONCENTRATION = 80.0
PCR_MAX_COPIES = 10**12

# Фрагментация и отбор библиотеки.
#
# Важное отличие от прежней реализации: число PCR1-копий НЕ обязано
# сохраняться как число фрагментов. Сначала моделируется конечный aliquot
# молекул из огромного PCR1-пула, затем каждая выбранная молекула физически
# разрезается. До size-selection проверяется сохранение нуклеотидной массы.
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "uniform_single_cut"

# Вычислительный library-input bottleneck: размер aliquot равен суммарному
# числу исходных молекул до PCR1, умноженному на этот коэффициент, но состав
# aliquot выбирается пропорционально abundance после PCR1. Так мы сохраняем
# PCR1-induced abundance без перебора триллионов копий.
LIBRARY_INPUT_SCALE = 1.0

# После разрезания выполняется отдельный мягкий size-selection. Параметры
# 200/40 теперь описывают отбор библиотеки, а не прямую генерацию длины.
SIZE_SELECTION_TARGET = 200
SIZE_SELECTION_SD = 40
MIN_SEQUENCABLE_FRAGMENT_LENGTH = 151

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "library_input": "finite aliquot sampled from PCR1 pool proportional to PCR1 abundance",
    "library_input_scale_vs_starting_copies": LIBRARY_INPUT_SCALE,
    "breakpoints": "exactly one uniformly sampled inter-base bond per library-input molecule",
    "daughter_fragments": "both physical daughters are created before size selection",
    "size_selection": {
        "type": "Gaussian acceptance kernel",
        "target_nt": SIZE_SELECTION_TARGET,
        "sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    },
    "scientific_analogs": [
        "BEERS2: default sequence-independent uniform bond breaking + separate size selection; DOI 10.1093/bib/bbae164",
        "Flux Simulator: modular fragmentation + size selection; DOI 10.1093/nar/gks666",
    ],
}

# PCR2: высокий предел предотвращает искусственное выравнивание численности.
LIBRARY_PCR_CYCLES = 10
LIBRARY_PCR_EFFICIENCY_MEAN = 0.85
LIBRARY_PCR_EFFICIENCY_CONCENTRATION = 80.0
LIBRARY_PCR_MAX_COPIES = 10**15

READ_BUDGET_MODE = "match_all_raw_pairs"
FIXED_READ_PAIRS = 500_000
MIXTURE_MODE = "observed_locus_depth"
RAW_PAIR_COUNTS = {
    "ERR3004229": 1_355_378,
    "ERR3004230": 1_153_931,
    "ERR3004231": 958_261,
    "ERR3004232": 1_085_144,
}

SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "InSilicoSeq bundled NovaSeq profile",
    "conditioned_on_post_annotation_filter": False,
}

print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# ---------------------------------------------------------------------
# Очистка сгенерированных артефактов в целевой ветви
# ---------------------------------------------------------------------
# Каталог выходных данных:
#   results/PRJEB30386/simulated/insilicoseq/
#
# При FORCE=True и CLEAN_OLD_OUTPUTS=True содержимое целевых
# выходных каталогов удаляется и создаётся заново. Исходные данные
# за пределами OUT_BASE не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [TRUTH_DIR,PCR1_DIR,FRAGMENTATION_DIR,PCR2_DIR,ALLOCATION_DIR,FASTQ_NATIVE_DIR,FASTQ_DIR,MODEL_DIR,QC_DIR,LOGS_DIR]

    for directory in generated_dirs:
        directory = Path(directory)
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        Path(directory).mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение эталона с учётом UMI V…J после фильтрации

`post_annotation_filtered/airr_pass` служит эталонным входом. Условия фильтра повторно проверяются утверждениями, а не отдельной политикой фильтрации. Дополнительно исключаются только непригодные UMI и записи с неоднозначными N.


In [ ]:
def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _barcode(sequence_id):
    m = re.search(r"(?:^|\|)BARCODE=([^|\s]+)", sequence_id or "")
    return m.group(1) if m else None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as src:
        for chunk in iter(lambda: src.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(r, run):
    """Fail loudly if airr_pass no longer matches the recorded filter contract."""
    problems = []
    expected_locus = RUN_LOCUS[run]

    v_start = _as_int(r.get("v_sequence_start"))
    j_end = _as_int(r.get("j_sequence_end"))
    vgs = _as_int(r.get("v_germline_start"))
    vi = _as_float(r.get("v_identity"))
    ji = _as_float(r.get("j_identity"))
    vs = _as_float(r.get("v_support"))
    js = _as_float(r.get("j_support"))

    if v_start is None or j_end is None or j_end - v_start + 1 < MIN_TEMPLATE_LENGTH:
        problems.append("V-J span")
    if vgs != 1:
        problems.append("V 5-prime completeness")
    if not _truthy(r.get("complete_vdj")):
        problems.append("complete_vdj")
    if str(r.get("locus", "")).strip() != expected_locus:
        problems.append("locus")
    if not _truthy(r.get("productive")):
        problems.append("productive")
    if not _falsey(r.get("stop_codon")):
        problems.append("stop_codon")
    if vi is None or vi < MIN_V_IDENTITY:
        problems.append("v_identity")
    if ji is None or ji < MIN_J_IDENTITY:
        problems.append("j_identity")
    if vs is None or vs > MAX_V_SUPPORT:
        problems.append("v_support")
    if js is None or js > MAX_J_SUPPORT:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{run} airr_pass violates filter_summary contract for "
            f"{r.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    data = {
        "dataset": DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "require_valid_20nt_umi": REQUIRE_VALID_20NT_UMI,
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "crop_to_annotated_V_to_J": True,
            "collapse_key": ["locus", "VJ_sequence"],
            "starting_copies_mode": STARTING_COPIES_MODE,
        },
        "branch": BRANCH_NAME,
        "seed": SEED,
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
    }
    return data

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False
    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and CLEAN_OLD_OUTPUTS=True "
            "or choose a new branch."
        )
    return True

def build_mixed_templates(force=FORCE):
    sample = MIXED_SAMPLE
    out_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    out_tsv = TEMPLATES_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / "source_truth_audit_by_run.tsv"

    if out_fa.exists() and out_tsv.exists() and audit_path.exists() and not force:
        _check_cached_truth_provenance()
        print("[skip] post-filter UMI-aware templates exist")
        return

    groups = {}
    audit = []

    for run in SOURCE_RUNS:
        airr = TRUTH_AIRR_DIR / f"{run}.airr.tsv"
        filtered_fastq = FILTERED_FASTQ_DIR / f"{run}_filtered.fastq.gz"
        if not airr.exists():
            raise FileNotFoundError(f"Filtered AIRR missing: {airr}")
        if not filtered_fastq.exists():
            raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")

        expected_passed = int(FILTER_SUMMARY["samples"][run]["passed"])
        a = {
            "source_run": run,
            "library": RUN_LIBRARY[run],
            "expected_locus": RUN_LOCUS[run],
            "expected_passed_from_filter_summary": expected_passed,
            "airr_pass_rows": 0,
            "simulation_eligible_rows": 0,
            "missing_barcode": 0,
            "invalid_barcode_length": 0,
            "non_acgt_barcode": 0,
            "missing_v_or_j_call": 0,
            "sequence_with_N": 0,
        }

        with open(airr, newline="") as h:
            for r in csv.DictReader(h, delimiter="\t"):
                a["airr_pass_rows"] += 1
                _validate_post_filter_row(r, run)

                start = int(r["v_sequence_start"]) - 1
                end = int(r["j_sequence_end"])
                seq = (r.get("sequence") or "")[start:end].upper()

                # Основной фильтр гарантирует длину не менее MIN_TEMPLATE_LENGTH.
                if len(seq) < MIN_TEMPLATE_LENGTH:
                    raise RuntimeError(
                        f"{run}: cropped V-J sequence shorter than filter threshold "
                        f"for {r.get('sequence_id')}"
                    )

                if not r.get("v_call") or not r.get("j_call"):
                    a["missing_v_or_j_call"] += 1
                    continue

                if DROP_SEQUENCES_WITH_N and "N" in seq:
                    a["sequence_with_N"] += 1
                    continue

                bc = _barcode(r.get("sequence_id"))
                if REQUIRE_VALID_20NT_UMI:
                    if not bc:
                        a["missing_barcode"] += 1
                        continue
                    if len(bc) != 20:
                        a["invalid_barcode_length"] += 1
                        continue
                    if set(bc.upper()) - set("ACGT"):
                        a["non_acgt_barcode"] += 1
                        continue

                a["simulation_eligible_rows"] += 1
                locus = r["locus"]
                key = (locus, seq)
                g = groups.setdefault(
                    key,
                    {
                        "observed_reads": 0,
                        "umis": set(),
                        "runs": set(),
                        "libraries": set(),
                        "v_calls": set(),
                        "j_calls": set(),
                    },
                )
                g["observed_reads"] += 1
                g["runs"].add(run)
                g["libraries"].add(RUN_LIBRARY[run])
                g["v_calls"].add(r["v_call"])
                g["j_calls"].add(r["j_call"])
                if bc:
                    g["umis"].add(run + ":" + bc)

        if a["airr_pass_rows"] != expected_passed:
            raise RuntimeError(
                f"{run}: airr_pass rows={a['airr_pass_rows']:,}, but "
                f"filter_summary passed={expected_passed:,}"
            )
        audit.append(a)

    ordered = sorted(
        groups.items(),
        key=lambda x: (
            x[0][0],
            -len(x[1]["umis"]),
            -x[1]["observed_reads"],
            x[0][1],
        ),
    )

    fields = [
        "template_id", "locus", "length", "observed_multiplicity",
        "unique_umis", "source_runs", "libraries", "v_calls", "j_calls",
    ]
    with open(out_fa, "w") as fa, open(out_tsv, "w", newline="") as ts:
        w = csv.DictWriter(ts, fieldnames=fields, delimiter="\t")
        w.writeheader()
        for i, ((locus, seq), g) in enumerate(ordered, 1):
            tid = f"{sample}_{locus}_tpl_{i:07d}"
            fa.write(f">{tid}\n{seq}\n")
            w.writerow({
                "template_id": tid,
                "locus": locus,
                "length": len(seq),
                "observed_multiplicity": g["observed_reads"],
                "unique_umis": len(g["umis"]),
                "source_runs": ";".join(sorted(g["runs"])),
                "libraries": ";".join(sorted(g["libraries"])),
                "v_calls": ";".join(sorted(g["v_calls"])),
                "j_calls": ";".join(sorted(g["j_calls"])),
            })

    with open(audit_path, "w", newline="") as h:
        fields = list(audit[0])
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows(audit)

    PROVENANCE_PATH.write_text(json.dumps(_current_provenance(), indent=2) + "\n")
    print(
        f"post-filter mixed templates={len(ordered):,}; "
        f"loci={sorted({k[0] for k in groups})}"
    )

build_mixed_templates()


## 5. Сводка контроля качества дедупликации

In [ ]:
q=TEMPLATES_DIR/f"{MIXED_SAMPLE}_template_qc.tsv"; rows=list(csv.DictReader(open(q),delimiter="\t")); loci=sorted({r["locus"] for r in rows}); summary=[]
for locus in loci:
    x=[r for r in rows if r["locus"]==locus]; summary.append({"locus":locus,"unique_templates":len(x),"observed_reads":sum(int(r["observed_multiplicity"]) for r in x),"unique_umis":sum(int(r["unique_umis"]) for r in x)})
out=QC_DIR/"template_summary_by_locus.tsv"
with open(out,"w",newline="") as h:
    w=csv.DictWriter(h,fieldnames=list(summary[0]),delimiter="\t"); w.writeheader(); w.writerows(summary)
print(*summary,sep="\n"); print("wrote",out)


## 6. Готовая модель ISS NovaSeq

Встроенный KDE-профиль NovaSeq из ISS 2.0.1 создаёт риды длиной 151 nt. После стадий эталон → PCR1 → фрагментация → PCR2 генерируются PE151, затем последовательности и качества детерминированно обрезаются до PE150. Пользовательская модель не обучается.

In [ ]:
import iss
ISS_PROFILE_NAME="NovaSeq"; ISS_NATIVE_READ_LENGTH=151
profile_path=Path(iss.__file__).resolve().parent/"profiles"/ISS_PROFILE_NAME
with np.load(profile_path,allow_pickle=True) as z:
    assert int(z["read_length"])==ISS_NATIVE_READ_LENGTH
    print({"profile":ISS_PROFILE_NAME,"model":str(z["model"].item()),"native_read_length":int(z["read_length"]),"target":TARGET_READ_LENGTH})


## 7. Бюджет смешанного секвенирования

Один синтетический FASTQ содержит все локусы. Глубина по умолчанию равна сумме глубин четырёх исходных индексированных библиотек. Бюджеты локусов воспроизводят их вычислительное объединение; внутри локуса численность определяется UMI и моделью PCR. Нативное спаривание тяжёлой и лёгкой цепей не моделируется.


In [ ]:
def get_read_budget(sample):
    if READ_BUDGET_MODE=="fixed": return int(FIXED_READ_PAIRS)
    if READ_BUDGET_MODE=="match_all_raw_pairs":
        total=0
        for run in SOURCE_RUNS:
            n1=count_fastq(RAW_FASTQ_DIR/f"{run}_1.fastq.gz"); n2=count_fastq(RAW_FASTQ_DIR/f"{run}_2.fastq.gz")
            if n1!=n2: raise ValueError(f"{run}: mate count mismatch")
            total+=n1
        return total
    raise ValueError(READ_BUDGET_MODE)
READ_BUDGETS={MIXED_SAMPLE:get_read_budget(MIXED_SAMPLE)}; print(READ_BUDGETS)


## 8. PCR1 для наблюдаемых шаблонов V…J


In [ ]:
def load_dedup_templates(sample):
    fa=TEMPLATES_DIR/f"{sample}_templates.fasta"; qc=TEMPLATES_DIR/f"{sample}_template_qc.tsv"; meta={r["template_id"]:r for r in csv.DictReader(open(qc),delimiter="\t")}
    return [{"template_id":tid,"sequence":seq,"length":len(seq),"observed_multiplicity":int(meta[tid]["observed_multiplicity"]),"unique_umis":int(meta[tid]["unique_umis"]),"locus":meta[tid]["locus"]} for tid,seq in iter_fasta(fa)]
def starting_copies(row):
    if STARTING_COPIES_MODE=="unique_umis": return max(1,row["unique_umis"])
    if STARTING_COPIES_MODE=="one_per_unique": return 1
    if STARTING_COPIES_MODE=="observed_multiplicity": return row["observed_multiplicity"]
    raise ValueError(STARTING_COPIES_MODE)
def branching_pcr(n0,efficiency,cycles,rng):
    n=int(n0)
    for _ in range(cycles):
        if n<=0:return 0
        n+=int(rng.binomial(n,efficiency))
        if n>=PCR_MAX_COPIES:return int(PCR_MAX_COPIES)
    return n
def simulate_pcr_pool(sample):
    rng=np.random.default_rng(sample_seed(sample,1000)); rows=load_dedup_templates(sample); mean=PCR_EFFICIENCY_MEAN; conc=PCR_EFFICIENCY_CONCENTRATION; alpha,beta=(mean*conc,(1-mean)*conc) if mean<1 else (None,None)
    for r in rows:
        p=1. if mean==1 else float(rng.beta(alpha,beta)); n0=starting_copies(r); r.update(starting_copies=n0,pcr_efficiency=p,pcr_copies=branching_pcr(n0,p,PCR_CYCLES,rng))
    return rows
_test_rng=np.random.default_rng(1); assert branching_pcr(1,1.,10,_test_rng)==2**10; assert branching_pcr(1,0.,10,_test_rng)==1; print("branching PCR smoke tests: OK")


## 9. Фрагментация: baseline random-cut

Эта версия больше не создаёт фиксированное число «репрезентативных фрагментов» на шаблон.

1. Из огромного PCR1-пула выбирается конечный aliquot молекул пропорционально abundance после PCR1. Размер aliquot задаётся через `LIBRARY_INPUT_SCALE` относительно числа исходных молекул до PCR1.
2. Для каждой выбранной физической молекулы выбирается **ровно одна** межнуклеотидная связь с равной вероятностью среди всех `L-1` возможных позиций.
3. Молекула действительно делится на **два дочерних фрагмента**.
4. После физического разрыва выполняется отдельный мягкий size-selection вокруг 200 nt (`SD=40`).
5. В PCR2 переходят только фрагменты, прошедшие size-selection и достаточно длинные для используемого профиля секвенирования.

Математически это соответствует независимому uniform random cut каждой молекулы, но события агрегируются по одинаковым координатам разрыва, поэтому триллионы PCR1-копий не создаются как отдельные Python-объекты.

Научная мотивация: BEERS2 использует sequence-independent uniform bond breaking как базовый режим fragmentation и отделяет fragmentation от size selection (DOI `10.1093/bib/bbae164`). Flux Simulator также моделирует fragmentation и size selection как отдельные стадии (DOI `10.1093/nar/gks666`).


In [ ]:
def _size_selection_probability(length):
    """Probability that a physical fragment enters the PCR2 library."""
    length = int(length)
    if length < MIN_SEQUENCABLE_FRAGMENT_LENGTH:
        return 0.0
    z = (length - SIZE_SELECTION_TARGET) / SIZE_SELECTION_SD
    return float(math.exp(-0.5 * z * z))


def _allocate_library_input(rows_pcr1, rng):
    """Sample a finite aliquot from the enormous PCR1 pool.

    The aliquot size is tied to pre-PCR molecular diversity (starting_copies),
    while template probabilities are determined by PCR1 output abundance.
    This keeps the simulation finite without flattening PCR1 effects.
    """
    starting_total = sum(max(0, int(r["starting_copies"])) for r in rows_pcr1)
    pcr1_total = sum(max(0, int(r["pcr_copies"])) for r in rows_pcr1)
    if starting_total <= 0 or pcr1_total <= 0:
        raise RuntimeError("empty PCR1 pool")

    requested = max(1, int(round(starting_total * LIBRARY_INPUT_SCALE)))
    n_input = min(requested, pcr1_total)
    weights = np.asarray([max(0, int(r["pcr_copies"])) for r in rows_pcr1], dtype=float)
    probs = weights / weights.sum()
    allocations = rng.multinomial(n_input, probs)
    return allocations, int(n_input), int(pcr1_total), int(starting_total)


def enumerate_fragments(sample, rows_pcr1, rng):
    """Uniform one-cut fragmentation of physical library-input molecules.

    Each selected molecule receives exactly one uniformly distributed cut at
    an inter-base bond. Both daughter molecules are created. A separate soft
    size-selection determines which daughters proceed to PCR2.
    """
    input_alloc, library_input_molecules, pcr1_total, starting_total = _allocate_library_input(rows_pcr1, rng)

    terminal = defaultdict(int)  # (template_id, locus, tlen, start, end) -> physical molecules
    input_nt_mass = 0
    library_input_templates = 0

    for r, n_input in zip(rows_pcr1, input_alloc):
        n_input = int(n_input)
        if n_input <= 0:
            continue
        library_input_templates += 1
        seq = r["sequence"]
        tlen = int(r["length"])
        if tlen < 2:
            continue

        input_nt_mass += n_input * tlen

        # Equivalent to cutting each physical molecule independently, but
        # aggregated over the L-1 possible inter-base bonds.
        cut_counts = rng.multinomial(
            n_input,
            np.full(tlen - 1, 1.0 / (tlen - 1), dtype=float),
        )
        for cut0 in np.flatnonzero(cut_counts):
            n = int(cut_counts[cut0])
            cut = int(cut0) + 1
            terminal[(r["template_id"], r["locus"], tlen, 0, cut)] += n
            terminal[(r["template_id"], r["locus"], tlen, cut, tlen)] += n

    terminal_fragment_molecules = sum(terminal.values())
    terminal_nt_mass = sum((end - start) * n for (_, _, _, start, end), n in terminal.items())
    if terminal_nt_mass != input_nt_mass:
        raise AssertionError(
            f"fragmentation nucleotide-mass conservation failed: input={input_nt_mass}, terminal={terminal_nt_mass}"
        )

    fragments = []
    retained_fragment_molecules = 0
    discarded_fragment_molecules = 0

    seq_by_id = {r["template_id"]: r["sequence"] for r in rows_pcr1}
    for (template_id, locus, tlen, start, end), n in terminal.items():
        flen = end - start
        p_keep = _size_selection_probability(flen)
        kept = int(rng.binomial(int(n), p_keep)) if p_keep > 0 else 0
        discarded_fragment_molecules += int(n) - kept
        if kept <= 0:
            continue
        retained_fragment_molecules += kept
        fragments.append({
            "fragment_id": f"{template_id}_rc_{start}_{end}",
            "template_id": template_id,
            "locus": locus,
            "template_length": int(tlen),
            "fragment_start": int(start),
            "fragment_length": int(flen),
            "sequence": seq_by_id[template_id][start:end],
            "fragment_input_copies": int(kept),
        })

    if not fragments:
        raise RuntimeError(f"{sample}: size selection removed the entire fragmented library")

    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_molecules_total": int(pcr1_total),
        "starting_copies_total": int(starting_total),
        "library_input_molecules": int(library_input_molecules),
        "library_input_templates": int(library_input_templates),
        "fragmentation_events": int(library_input_molecules),
        "terminal_fragment_molecules": int(terminal_fragment_molecules),
        "terminal_fragment_species": int(len(terminal)),
        "retained_fragment_molecules": int(retained_fragment_molecules),
        "retained_fragment_species": int(len(fragments)),
        "discarded_fragment_molecules": int(discarded_fragment_molecules),
        "input_nt_mass": int(input_nt_mass),
        "terminal_nt_mass": int(terminal_nt_mass),
        "fragmentation_nt_conserved": bool(input_nt_mass == terminal_nt_mass),
        "size_selection_target_nt": int(SIZE_SELECTION_TARGET),
        "size_selection_sd_nt": int(SIZE_SELECTION_SD),
        "min_sequencable_fragment_nt": int(MIN_SEQUENCABLE_FRAGMENT_LENGTH),
    }
    return fragments, summary


## 10. PCR2 для фрагментов

PCR2 применяется к сохранённым фрагментам и формирует отдельную таблицу стадии.


In [ ]:
def branching_pcr_vectorized(n0, efficiency, cycles, max_copies, rng):
    n = np.asarray(n0, dtype=np.int64).copy()
    eff = np.asarray(efficiency, dtype=np.float64)
    for _ in range(cycles):
        active = n > 0
        if not active.any():
            break
        new = np.zeros_like(n)
        new[active] = rng.binomial(n[active], eff[active])
        n = n + new
        np.minimum(n, max_copies, out=n)
    return n


def run_pcr2_on_fragments(fragments, rng):
    mean = LIBRARY_PCR_EFFICIENCY_MEAN
    conc = LIBRARY_PCR_EFFICIENCY_CONCENTRATION
    if mean == 1:
        efficiency = np.ones(len(fragments))
    else:
        alpha, beta = mean * conc, (1 - mean) * conc
        efficiency = rng.beta(alpha, beta, size=len(fragments))

    n0 = np.array([f["fragment_input_copies"] for f in fragments], dtype=np.int64)
    copies = branching_pcr_vectorized(
        n0, efficiency, LIBRARY_PCR_CYCLES, LIBRARY_PCR_MAX_COPIES, rng
    )
    for f, eff, cp in zip(fragments, efficiency, copies):
        f["library_pcr_efficiency"] = float(eff)
        f["library_pcr2_copies"] = int(cp)
    return fragments

_test_rng = np.random.default_rng(1)
_vec = branching_pcr_vectorized(
    np.array([1]), np.array([1.0]), 10, 10**15, _test_rng
)
assert int(_vec[0]) == 2**10
print("vectorized branching PCR smoke test: OK")


## 11. Сохранение PCR1 → фрагментация → PCR2 → точное распределение ридов


In [ ]:
def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows({k: r[k] for k in fields} for r in rows)
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR1", out)
        return
    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id", "locus", "length", "observed_multiplicity",
        "unique_umis", "starting_copies", "pcr_efficiency", "pcr_copies",
    ]
    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)

def run_fragmentation_stage(sample, force=FORCE):
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out = FRAGMENTATION_DIR / f"{sample}_fragments.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_fragmentation_summary.tsv"

    if out.exists() and summary_out.exists() and not force:
        print("[skip] fragmentation", out)
        return

    rows = []
    for row in read_dict_rows(inp):
        rows.append({
            **row,
            "length": int(row["length"]),
            "starting_copies": int(row["starting_copies"]),
            "pcr_copies": int(row["pcr_copies"]),
            "sequence": None,
        })

    sequences = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    for row in rows:
        row["sequence"] = sequences[row["template_id"]]

    fragments, frag_summary = enumerate_fragments(
        sample,
        rows,
        np.random.default_rng(sample_seed(sample, 2000)),
    )

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "sequence", "fragment_input_copies",
    ]
    write_dict_rows(out, fragments, fields)
    write_dict_rows(summary_out, [frag_summary], list(frag_summary))

    print("fragmentation model:", FRAGMENTATION_MODEL)
    print("fragmentation rows:", len(fragments), "->", out)
    print("fragmentation summary:", frag_summary)


def run_pcr2_stage(sample, force=FORCE):
    inp = FRAGMENTATION_DIR / f"{sample}_fragments.tsv"
    out = PCR2_DIR / f"{sample}_pcr2_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR2", out)
        return

    frags = read_dict_rows(inp)
    for f in frags:
        for k in (
            "template_length", "fragment_start", "fragment_length",
            "fragment_input_copies",
        ):
            f[k] = int(f[k])

    frags = run_pcr2_on_fragments(
        frags,
        np.random.default_rng(sample_seed(sample, 3000)),
    )
    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "sequence", "fragment_input_copies",
        "library_pcr_efficiency", "library_pcr2_copies",
    ]
    write_dict_rows(out, frags, fields)
    print("PCR2 rows:", len(frags), "->", out)

def run_allocation_stage(sample, force=FORCE):
    inp = PCR2_DIR / f"{sample}_pcr2_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fa = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"
    if out_counts.exists() and out_fa.exists() and out_qc.exists() and not force:
        print("[skip] allocation")
        return

    frags = read_dict_rows(inp)
    for f in frags:
        for k in (
            "template_length", "fragment_start", "fragment_length",
            "fragment_input_copies", "library_pcr2_copies",
        ):
            f[k] = int(f[k])
        f["library_pcr_efficiency"] = float(f["library_pcr_efficiency"])

    weights = np.asarray([f["library_pcr2_copies"] for f in frags], dtype=float)
    if not len(weights) or weights.sum() <= 0:
        raise RuntimeError("PCR2 pool is empty")

    target = int(READ_BUDGETS[sample])
    allocated = np.zeros(len(frags), dtype=np.int64)
    rng = np.random.default_rng(sample_seed(sample, 4000))

    if MIXTURE_MODE == "umi_pool":
        allocated = rng.multinomial(target, weights / weights.sum())
    elif MIXTURE_MODE == "observed_locus_depth":
        depth = {
            l: sum(
                RAW_PAIR_COUNTS[r]
                for r in SOURCE_RUNS
                if RUN_LOCUS[r] == l
            )
            for l in ("IGH", "IGK", "IGL")
        }
        exact = {l: target * n / sum(depth.values()) for l, n in depth.items()}
        budget = {l: int(np.floor(v)) for l, v in exact.items()}
        for l in sorted(
            exact,
            key=lambda x: exact[x] - budget[x],
            reverse=True,
        )[: target - sum(budget.values())]:
            budget[l] += 1

        for l, n in budget.items():
            idx = np.asarray([i for i, f in enumerate(frags) if f["locus"] == l])
            if not len(idx):
                raise RuntimeError(f"No simulated fragments for locus {l}")
            lw = weights[idx]
            allocated[idx] = rng.multinomial(n, lw / lw.sum())
        print("locus budgets", budget)
    else:
        raise ValueError(MIXTURE_MODE)

    with open(Path(str(out_counts) + ".tmp"), "w", newline="") as ch, \
         open(Path(str(out_fa) + ".tmp"), "w") as fh:
        w = csv.writer(ch, delimiter="\t")
        for f, n in zip(frags, allocated):
            f["simulated_read_pairs"] = int(n)
            if n:
                # InSilicoSeq readcount_file ожидает число ридов, поэтому число пар умножается на 2.
                w.writerow([f["fragment_id"], int(n) * 2])
                fh.write(f">{f['fragment_id']}\n{f['sequence']}\n")

    Path(str(out_counts) + ".tmp").replace(out_counts)
    Path(str(out_fa) + ".tmp").replace(out_fa)

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "fragment_input_copies",
        "library_pcr_efficiency", "library_pcr2_copies",
        "simulated_read_pairs",
    ]
    write_dict_rows(out_qc, frags, fields)
    assert int(allocated.sum()) == target
    print("allocated", target, "pairs")

for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_fragmentation_stage(sample)
    run_pcr2_stage(sample)
    run_allocation_stage(sample)


## 12. Контроль качества PCR1, фрагментации и PCR2


In [ ]:
pcr_summary = []
for sample in SAMPLES:
    pcr1_rows = read_dict_rows(PCR1_DIR / f"{sample}_pcr1_pool.tsv")
    frag_rows = read_dict_rows(FRAGMENTATION_DIR / f"{sample}_fragments.tsv")
    frag_summary = read_dict_rows(FRAGMENTATION_DIR / f"{sample}_fragmentation_summary.tsv")[0]
    pcr2_rows = read_dict_rows(PCR2_DIR / f"{sample}_pcr2_pool.tsv")

    pcr1_n = len(pcr1_rows)
    pcr1_molecules = sum(int(r["pcr_copies"]) for r in pcr1_rows)
    fragment_input_molecules = sum(int(r["fragment_input_copies"]) for r in frag_rows)

    if str(frag_summary["fragmentation_nt_conserved"]).lower() not in {"true", "1"}:
        raise AssertionError("fragmentation nucleotide-mass conservation QC failed")
    if int(frag_summary["retained_fragment_molecules"]) != fragment_input_molecules:
        raise AssertionError("fragmentation retained-molecule accounting QC failed")

    copies2 = [int(r["library_pcr2_copies"]) for r in pcr2_rows]
    eff2 = [float(r["library_pcr_efficiency"]) for r in pcr2_rows]

    allocated_n = allocated_nonzero = allocated_total = 0
    with open(ALLOCATION_DIR / f"{sample}_allocation.tsv") as h:
        for r in csv.DictReader(h, delimiter="\t"):
            n = int(r["simulated_read_pairs"])
            allocated_n += 1
            allocated_total += n
            allocated_nonzero += int(n > 0)

    pcr_summary.append({
        "sample": sample,
        "fragmentation_model": frag_summary["fragmentation_model"],
        "pcr1_templates": pcr1_n,
        "pcr1_molecules": pcr1_molecules,
        "library_input_molecules": int(frag_summary["library_input_molecules"]),
        "fragmentation_events": int(frag_summary["fragmentation_events"]),
        "terminal_fragment_molecules": int(frag_summary["terminal_fragment_molecules"]),
        "terminal_fragment_species": int(frag_summary["terminal_fragment_species"]),
        "retained_fragment_molecules": fragment_input_molecules,
        "retained_fragment_species": len(frag_rows),
        "discarded_fragment_molecules": int(frag_summary["discarded_fragment_molecules"]),
        "input_nt_mass": int(frag_summary["input_nt_mass"]),
        "terminal_nt_mass": int(frag_summary["terminal_nt_mass"]),
        "fragmentation_nt_conserved": True,
        "pcr2_fragments": len(copies2),
        "target_read_pairs": allocated_total,
        "fragments_with_reads": allocated_nonzero,
        "fragment_sampling_dropout": 1 - allocated_nonzero / allocated_n,
        "mean_pcr2_efficiency": float(np.mean(eff2)),
        "median_pcr2_copies": float(np.median(copies2)),
    })

out = QC_DIR / "pcr1_fragmentation_pcr2_summary.tsv"
write_dict_rows(out, pcr_summary, list(pcr_summary[0]))
print(*pcr_summary, sep="\n")


## 13. Генерация ридов точной длины PE150


In [ ]:
def crop_fastq(in_path,out_path,length):
    tmp=Path(str(out_path)+".tmp")
    with open_text(in_path,"rt") as src,gzip.open(tmp,"wt") as dst:
        n=0
        while True:
            h=src.readline()
            if not h: break
            s=src.readline().rstrip("\n\r"); p=src.readline(); q=src.readline().rstrip("\n\r")
            if len(s)<length or len(q)<length: raise ValueError(f"short read in {in_path}: {len(s)}")
            dst.write(h); dst.write(s[:length]+"\n"); dst.write(p); dst.write(q[:length]+"\n"); n+=1
    tmp.replace(out_path); return n

def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; native_prefix=FASTQ_NATIVE_DIR/sample; final_prefix=FASTQ_DIR/sample; ext=".fastq.gz"
    nr1=Path(str(native_prefix)+f"_R1{ext}"); nr2=Path(str(native_prefix)+f"_R2{ext}"); r1=Path(str(final_prefix)+f"_R1{ext}"); r2=Path(str(final_prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",ISS_PROFILE_NAME,"--cpus",str(NPROC),"--output",str(native_prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_novaseq.log")
    n1=crop_fastq(nr1,r1,TARGET_READ_LENGTH); n2=crop_fastq(nr2,r2,TARGET_READ_LENGTH); assert n1==n2==READ_BUDGETS[sample]
    print("NovaSeq native PE151 -> exact PE150:",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":READ_BUDGETS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==READ_BUDGETS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и научные ограничения

- Биологический эталон формируется из AIRR-записей после фильтрации аннотаций.
- Сохраняется наблюдаемая последовательность V…J; germline-последовательность её не заменяет, поэтому реальные SHM остаются в данных.
- Численность UMI задаёт исходные копии. Записи с отсутствующим или некорректным UMI и неоднозначными N исключаются и учитываются отдельно.
- PCR1 и PCR2 моделируются стохастическим ветвлением. Замены полимеразы, химеры, истощение реагентов и изменение эффективности по циклам явно не моделируются.
- `observed_locus_depth` сохраняет наблюдаемую долю IGH/IGK/IGL.
- Встроенная модель InSilicoSeq NovaSeq является общим профилем платформы: ISS создаёт PE151, после чего оба рида детерминированно обрезаются до PE150.
- Смешанный IGH+IGK+IGL набор является вычислительным тестом без нативного спаривания тяжёлой и лёгкой цепей.
- Фрагментация использует один равновероятный random cut на выбранную физическую молекулу; оба дочерних фрагмента создаются до size-selection.
- Между PCR1 и fragmentation явно моделируется конечный library-input aliquot; до size-selection проверяется сохранение суммарной нуклеотидной массы.
